In [ ]:
%pip install -U geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 70.2 MB/s  0:00:00


In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
#authenticate and initalize ee
ee.Authenticate()
ee.Initialize(project='my-project11-429606')



In [ ]:

# Load administrative boundaries
admin = ee.FeatureCollection(
    "FAO/GAUL/2015/level2"
)

# Filter Nigeria
nigeria = admin.filter(
    ee.Filter.eq("ADM0_NAME", "Nigeria")
)

# Inspect the state names
states = nigeria.aggregate_array(
    "ADM1_NAME"
).distinct().sort()

print("States in the dataset:")
print(states.getInfo())


In [ ]:

# Extract Lagos State
lagos = nigeria.filter(
    ee.Filter.eq("ADM1_NAME", "Lagos")
)

# Extract its second-level administrative units
lagos_lgas = lagos

# Count the administrative units
lga_count = lagos_lgas.size().getInfo()

# Get LGA names
lga_names = (
    lagos_lgas
    .aggregate_array("ADM2_NAME")
    .distinct()
    .sort()
    .getInfo()
)

print("Number of administrative units:", lga_count)

print("\nLGA names:")
for name in lga_names:
    print(name)


In [ ]:

# Create an interactive map
Map = geemap.Map(basemap="SATELLITE",
    center=[6.5, 3.4],
    zoom=9
)

# Style the LGA boundaries
lga_style = lagos_lgas.style(
    color="red",
    fillColor="00000000",
    width=2
)

# Add the boundaries
Map.addLayer(
    lga_style,
    {},
    "Lagos LGA Boundaries"
)

# Display the map
Map


In [ ]:


#Count administrative units
lga_count = lagos_lgas.size().getInfo()

#Retrieve LGA names
lga_names = (
    lagos_lgas
    .aggregate_array("ADM2_NAME")
    .distinct()
    .sort()
    .getInfo()
)

#Display count
print(f"\nTotal number of LGAs: {lga_count}")

#Display names with numbering
print("\nLGA NAMES:")

for number, name in enumerate(lga_names, start=1):
    print(f"{number:02d}. {name}")




We will use the 2023 wet-season period (May–October) as the test year. Once the workflow works, we can apply it to the other seven years.

In [ ]:
# Define the Lagos study area
lagos_geometry = lagos_lgas.geometry()

# Define observation period(May 2023-Nov 2023)
start_date = "2023-05-01"
end_date = "2023-11-01"  # End date is exclusive


In [ ]:
# Load Sentinel-2 Surface Reflectance
s2_raw = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(lagos_geometry)
    .filterDate(start_date, end_date)
    .filter(
        ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 90)
    )
)


In [ ]:
# Inspect collection size
image_count = s2_raw.size().getInfo()

print("SENTINEL-2 DATA INSPECTION")
print("Study area: Lagos State")
print("Observation period: May - October 2023")
print("Images available:", image_count)

In [ ]:
# CLOUD AND SHADOW MASKING

def mask_sentinel2(image):

    scl = image.select("SCL")

    clear_mask = (
        scl.neq(1)
        .And(scl.neq(2))
        .And(scl.neq(3))
        .And(scl.neq(7))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    # Convert surface reflectance to 0-1 scale
    reflectance = (
        image.select(["B2", "B3", "B4", "B8"])
        .multiply(0.0001)
    )

    # Apply mask and retain original image properties
    return (
        image.addBands(
            reflectance,
            overwrite=True
        )
        .updateMask(clear_mask)
        .copyProperties(image, ["system:time_start"])
    )


In [ ]:
# Apply masking to the collection
s2_clean = s2_raw.map(mask_sentinel2)


In [ ]:
def add_vegetation_indices(image):

    # NDVI = (NIR - Red) / (NIR + Red)
    ndvi = (
        image.normalizedDifference(["B8", "B4"])
        .rename("NDVI")
    )

    # EVI = 2.5 * (NIR - Red) /
    #       (NIR + 6*Red - 7.5*Blue + 1)

    evi = (
        image.expression(
            "2.5 * ((NIR - RED) / "
            "(NIR + 6 * RED - 7.5 * BLUE + 1))",
            {
                "NIR": image.select("B8"),
                "RED": image.select("B4"),
                "BLUE": image.select("B2")
            }
        )
        .rename("EVI")
    )

    return image.addBands([ndvi, evi])


s2_indices = s2_clean.map(add_vegetation_indices)



In [ ]:
# 2023 WET-SEASON MEDIAN COMPOSITE

composite_2023 = (
    s2_indices
    .median()
    .clip(lagos_geometry)
)

# Select vegetation indices
ndvi_2023 = composite_2023.select("NDVI")
evi_2023 = composite_2023.select("EVI")



In [ ]:
import geemap.core as geemap

In [ ]:
Map = geemap.Map()

# Centre the map on Lagos State
Map.centerObject(lagos_geometry, 9)

# NDVI colour palette
ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": [
        "#a50026",
        "#fdae61",
        "#ffffbf",
        "#a6d96a",
        "#1a9850"
    ]
}

# Add NDVI layer
Map.addLayer(
    ndvi_2023,
    ndvi_vis,
    "NDVI | May–October 2023"
)

# Style LGA boundaries
lga_outline = lagos_lgas.style(
    color="#555555",
    fillColor="00000000",
    width=1
)

# Add administrative boundaries
Map.addLayer(
    lga_outline,
    {},
    "LGA boundaries"
)

# Display map
Map

In [ ]:
# Years for the analysis
years = ee.List.sequence(2019, 2025)

# Function to generate one annual composite
def make_annual_composite(year):

    year = ee.Number(year)

    start = ee.Date.fromYMD(year, 5, 1)
    end = ee.Date.fromYMD(year, 11, 1)

    # Filter Sentinel-2 imagery
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(lagos_geometry)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 90))
        .map(mask_sentinel2)
        .map(add_vegetation_indices)
    )

    # Generate median composite
    composite = (
        collection
        .select(["NDVI", "EVI"])
        .median()
        .clip(lagos_geometry)
        .set("year", year)
        .set("scene_count", collection.size())
        .set("system:time_start", start.millis())
    )

    return composite


# Build the annual ImageCollection
annual_composites = ee.ImageCollection.fromImages(
    years.map(make_annual_composite)
)

print("Annual composites created:", annual_composites.size().getInfo())

In [ ]:
# Retrieve annual metadata
composite_years = annual_composites.aggregate_array(
    "year"
).getInfo()

scene_counts = annual_composites.aggregate_array(
    "scene_count"
).getInfo()

# Display results
for year, count in zip(composite_years, scene_counts):
    print(f"{year}: {count} Sentinel-2 scenes")

In [ ]:
# Function to calculate mean NDVI and EVI for each LGA
def calculate_lga_statistics(image):

    year = image.get("year")

    statistics = image.select(["NDVI", "EVI"]).reduceRegions(
        collection=lagos_lgas,
        reducer=ee.Reducer.mean(),
        scale=10,
        tileScale=4
    )

    # Attach the year to each LGA record
    statistics = statistics.map(
        lambda feature: ee.Feature(feature).set("year", year)
    )

    return statistics


# Calculate statistics across all annual composites
lga_statistics = ee.FeatureCollection(
    annual_composites.map(calculate_lga_statistics)
).flatten()

print("Total LGA records:", lga_statistics.size().getInfo())

In [ ]:
# Display the first five records
sample_records = lga_statistics.limit(5).getInfo()

for feature in sample_records["features"]:
    print(feature["properties"])


export_task = ee.batch.Export.table.toDrive(
    collection=lga_statistics,
    description="Lagos_LGA_NDVI_EVI_2018_2025",
    folder="Lagos_Vegetation_Monitoring",
    fileNamePrefix="Lagos_LGA_NDVI_EVI_2018_2025",
    fileFormat="CSV",
    selectors=[
        "ADM2_NAME",
        "year",
        "NDVI",
        "EVI"
    ]
)

export_task.start()

print("Export started.")
print(export_task.status())

In [ ]:
print(export_task.status())

In [ ]:
# Calculate annual mean NDVI and EVI for Lagos State

def calculate_state_statistics(image):

    year = image.get("year")

    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=lagos_geometry,
        scale=10,
        maxPixels=1e13,
        tileScale=4
    )

    return ee.Feature(None, {
        "year": year,
        "NDVI": stats.get("NDVI"),
        "EVI": stats.get("EVI")
    })


# Generate state-wide annual statistics
state_statistics = ee.FeatureCollection(
    annual_composites.map(calculate_state_statistics)
)

print("Annual records:", state_statistics.size().getInfo())

In [ ]:
state_results = state_statistics.sort("year").getInfo()

for feature in state_results["features"]:
    properties = feature["properties"]

    print(
        f"Year: {properties['year']} | "
        f"NDVI: {properties['NDVI']:.4f} | "
        f"EVI: {properties['EVI']:.4f}"
    )

Correct Evi calavulation

In [ ]:
def add_vegetation_indices(image):

    # Calculate NDVI
    ndvi = image.normalizedDifference(
        ["B8", "B4"]
    ).rename("NDVI")

    # Extract reflectance bands
    nir = image.select("B8")
    red = image.select("B4")
    blue = image.select("B2")

    # Calculate EVI denominator
    denominator = (
        nir.add(red.multiply(6))
        .subtract(blue.multiply(7.5))
        .add(1)
    )

    # Calculate EVI
    evi = image.expression(
        "2.5 * ((NIR - RED) / DENOM)",
        {
            "NIR": nir,
            "RED": red,
            "DENOM": denominator
        }
    ).rename("EVI")

    # Remove unstable denominator values and implausible EVI outliers
    valid_evi = (
        denominator.abs().gt(0.1)
        .And(evi.gte(-1))
        .And(evi.lte(1))
    )

    evi = evi.updateMask(valid_evi)

    return image.addBands(
        [ndvi, evi],
        overwrite=True
    )

In [ ]:
#Rebuild annaual composites

In [ ]:
years = ee.List.sequence(2019, 2025)

def make_annual_composite(year):

    year = ee.Number(year)

    start = ee.Date.fromYMD(year, 5, 1)
    end = ee.Date.fromYMD(year, 11, 1)

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(lagos_geometry)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 90))
        .map(mask_sentinel2)
        .map(add_vegetation_indices)
    )

    composite = (
        collection
        .select(["NDVI", "EVI"])
        .median()
        .clip(lagos_geometry)
        .set("year", year)
        .set("scene_count", collection.size())
        .set("system:time_start", start.millis())
    )

    return composite


annual_composites = ee.ImageCollection.fromImages(
    years.map(make_annual_composite)
)

print(
    "Annual composites:",
    annual_composites.size().getInfo()
)

In [ ]:
state_statistics = ee.FeatureCollection(
    annual_composites.map(calculate_state_statistics)
)

state_results = state_statistics.sort("year").getInfo()

for feature in state_results["features"]:

    properties = feature["properties"]

    print(
        f"Year: {properties['year']} | "
        f"NDVI: {properties['NDVI']} | "
        f"EVI: {properties['EVI']}"
    )

In [ ]:
# Recalculate LGA statistics using corrected annual composites

lga_statistics = ee.FeatureCollection(
    annual_composites.map(calculate_lga_statistics)
).flatten()

print(
    "Updated LGA records:",
    lga_statistics.size().getInfo()
)

# Check valid NDVI and EVI records
valid_records = lga_statistics.filter(
    ee.Filter.notNull(["NDVI", "EVI"])
)

print(
    "Records with valid indices:",
    valid_records.size().getInfo()
)

In [ ]:
lga_export = ee.batch.Export.table.toDrive(
    collection=lga_statistics,
    description="Lagos_LGA_NDVI_EVI_2018_2025_Final",
    folder="Lagos_Vegetation_Monitoring",
    fileNamePrefix="Lagos_LGA_NDVI_EVI_2018_2025_Final",
    fileFormat="CSV",
    selectors=[
        "ADM2_NAME",
        "year",
        "NDVI",
        "EVI"
    ]
)

lga_export.start()

print(lga_export.status())

In [ ]:
print(lga_export.status())

In [ ]:
print(lga_export.status())

Visualisation

In [ ]:
import numpy as np

# Retrieve the corrected LGA statistics
results = lga_statistics.getInfo()

# Convert Earth Engine results into a DataFrame
records = [
    feature["properties"]
    for feature in results["features"]
]

df = pd.DataFrame(records)

# Keep only the required columns
df = df[["ADM2_NAME", "year", "NDVI", "EVI"]]

# Convert data types
df["year"] = df["year"].astype(int)
df["NDVI"] = pd.to_numeric(df["NDVI"], errors="coerce")
df["EVI"] = pd.to_numeric(df["EVI"], errors="coerce")

print(df.head())
print("\nDataset dimensions:", df.shape)
print("\nMissing values:")
print(df[["NDVI", "EVI"]].isna().sum())

In [ ]:
# Keep the original dataset
df_clean = df.copy()

# Check missing values by year
print("Missing values by year:")

print(
    df_clean.groupby("year")[["NDVI", "EVI"]]
    .apply(lambda x: x.isna().sum())
)

# Confirm dataset dimensions
print("\nDataset dimensions:", df_clean.shape)

# Confirm missing records
print(
    "\nRecords with missing indices:",
    df_clean[df_clean[["NDVI", "EVI"]].isna().any(axis=1)]
    [["ADM2_NAME", "year", "NDVI", "EVI"]]
)

In [ ]:
# Identify LGAs with valid observations in both endpoint years

valid_lgas = (
    df[df["year"].isin([2019, 2025])]
    .groupby("ADM2_NAME")
    .filter(
        lambda x: (
            len(x) == 2 and
            x[["NDVI", "EVI"]].notna().all().all()
        )
    )["ADM2_NAME"].unique()
)

comparison_df = df[
    df["ADM2_NAME"].isin(valid_lgas)
].copy()

print("LGAs available for endpoint comparison:", len(valid_lgas))
print("Comparison dataset:", comparison_df.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

ndvi_pivot = comparison_df.pivot(
    index="ADM2_NAME",
    columns="year",
    values="NDVI"
).sort_index()

plt.figure(figsize=(12, 9))

sns.heatmap(
    ndvi_pivot,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    linewidths=0.5,
    vmin=-0.2,
    vmax=0.8
)

plt.title("Annual NDVI Distribution Across Lagos LGAs (2019–2025)")
plt.xlabel("Year")
plt.ylabel("LGA")
plt.tight_layout()
plt.show()

In [ ]:
evi_pivot = comparison_df.pivot(
    index="ADM2_NAME",
    columns="year",
    values="EVI"
).sort_index()

plt.figure(figsize=(12, 9))

sns.heatmap(
    evi_pivot,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    linewidths=0.5,
    vmin=-0.2,
    vmax=0.8
)

plt.title("Annual EVI Distribution Across Lagos LGAs (2019–2025)")
plt.xlabel("Year")
plt.ylabel("LGA")
plt.tight_layout()
plt.show()

In [ ]:
ndvi_change = (
    ndvi_pivot[2025] - ndvi_pivot[2019]
).sort_values()

evi_change = (
    evi_pivot[2025] - evi_pivot[2019]
).sort_values()

print("NDVI change (2019–2025):")
print(ndvi_change.round(4))

print("\nEVI change (2019–2025):")
print(evi_change.round(4))

In [ ]:
# Prepare state-wide annual statistics
state_df = pd.DataFrame(
    [
        feature["properties"]
        for feature in state_statistics.getInfo()["features"]
    ]
)

state_df["year"] = state_df["year"].astype(int)
state_df = state_df.sort_values("year")

# Plot annual NDVI and EVI trends
fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    state_df["year"],
    state_df["NDVI"],
    marker="o",
    label="NDVI"
)

ax.plot(
    state_df["year"],
    state_df["EVI"],
    marker="s",
    label="EVI"
)

ax.set_title("Annual Mean Vegetation Indices Across Lagos State (2019–2025)")
ax.set_xlabel("Year")
ax.set_ylabel("Mean Vegetation Index")
ax.set_xticks(range(2019, 2026))
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Prepare annual NDVI images for animation

ndvi_frames = annual_composites.map(
    lambda image: (
        image.select("NDVI")
        .visualize(
            min=-0.2,
            max=0.8,
            palette=[
                "#a50026",
                "#fdae61",
                "#ffffbf",
                "#a6d96a",
                "#1a9850"
            ]
        )
        .set(
            "system:time_start",
            image.get("system:time_start")
        )
    )
)

ndvi_frames = ndvi_frames.sort("system:time_start")

print("Animation frames:", ndvi_frames.size().getInfo())

In [ ]:
video_task = ee.batch.Export.video.toDrive(
    collection=ndvi_frames,
    description="Lagos_Annual_NDVI_Animation_2019_2025",
    folder="Lagos_Vegetation_Monitoring",
    fileNamePrefix="Lagos_NDVI_Animation_2019_2025",
    framesPerSecond=1,
    dimensions=1200,
    region=lagos_geometry
)

video_task.start()

print("Export status:", video_task.status()["state"])

In [ ]:

print("Export status:", video_task.status()["state"])

In [ ]:
import ee
import requests
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO

# Consistent NDVI visualisation
ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": [
        "#a50026",
        "#fdae61",
        "#ffffbf",
        "#a6d96a",
        "#1a9850"
    ]
}

# Get the 2019 annual NDVI image
image_2019 = ee.Image(
    annual_composites
    .filter(ee.Filter.eq("year", 2019))
    .first()
).select("NDVI")

# Render NDVI and LGA boundaries
rgb_2019 = (
    image_2019.visualize(**ndvi_vis)
    .blend(
        lagos_lgas.style(
            color="white",
            fillColor="00000000",
            width=1
        )
    )
)

# Generate a small  map image
url = rgb_2019.getThumbURL({
    "region": lagos_geometry,
    "dimensions": 1100,
    "format": "png"
})

response = requests.get(url)
response.raise_for_status()

map_image = Image.open(
    BytesIO(response.content)
).convert("RGB")

print("Map dimensions:", map_image.size)

In [ ]:
display(map_image)

In [ ]:
# Create a white background behind the NDVI map
white_background = ee.Image.constant(
    [255, 255, 255]
).byte()

rgb_2019_white = white_background.blend(rgb_2019)

# Generate thumbnail
url = rgb_2019_white.getThumbURL({
    "region": lagos_geometry.bounds(),
    "dimensions": 1100,
    "format": "png"
})

response = requests.get(url)
response.raise_for_status()

map_image = Image.open(
    BytesIO(response.content)
).convert("RGB")

print("Map dimensions:", map_image.size)

display(map_image)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import matplotlib.colors as mcolors

def create_presentation_frame(map_image, year):

    # Create presentation canvas
    canvas = Image.new("RGB", (1600, 900), "white")
    draw = ImageDraw.Draw(canvas)

    # Fonts
    try:
        title_font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 38
        )
        year_font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 58
        )
        label_font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 23
        )
        small_font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 18
        )
    except:
        title_font = year_font = label_font = small_font = ImageFont.load_default()

    # Main title
    title = "LAGOS STATE VEGETATION GREENNESS MONITORING"
    draw.text((800, 55), title, fill="#17324D",
              font=title_font, anchor="mt")

    draw.text((800, 115), "Annual NDVI Assessment | 2019–2025",
              fill="#555555", font=label_font, anchor="mt")

    # Dynamic year
    draw.text((800, 205), str(year), fill="#247A45",
              font=year_font, anchor="mt")

    # Resize map while preserving its proportions
    map_copy = map_image.copy()
    map_copy.thumbnail((1480, 390))

    x = (1600 - map_copy.width) // 2
    y = 290

    canvas.paste(map_copy, (x, y))

    # Legend heading
    draw.text((800, 735), "VEGETATION GREENNESS (NDVI)",
              fill="#17324D", font=label_font, anchor="mt")

    # Continuous colour gradient
    palette = [
        "#a50026", "#fdae61", "#ffffbf",
        "#a6d96a", "#1a9850"
    ]

    cmap = mcolors.LinearSegmentedColormap.from_list(
        "ndvi_palette", palette
    )

    bar_x, bar_y = 300, 785
    bar_width, bar_height = 1000, 28

    for i in range(bar_width):
        color = cmap(i / (bar_width - 1))
        rgb = tuple(int(v * 255) for v in color[:3])

        draw.line(
            (bar_x + i, bar_y, bar_x + i, bar_y + bar_height),
            fill=rgb
        )

    # NDVI scale values
    scale_values = [-0.2, 0.0, 0.2, 0.4, 0.6, 0.8]

    for value in scale_values:
        position = bar_x + int(
            ((value + 0.2) / 1.0) * bar_width
        )

        draw.text(
            (position, 825),
            str(value),
            fill="#333333",
            font=small_font,
            anchor="mt"
        )

    # Explanation
    draw.text(
        (800, 865),
        "Red/yellow: lower greenness  |  Green: higher greenness",
        fill="#555555",
        font=small_font,
        anchor="mt"
    )

    return canvas


# Create 2019 presentation preview
frame_2019 = create_presentation_frame(map_image, 2019)

display(frame_2019)

In [ ]:
from matplotlib import font_manager
from PIL import ImageFont

# Automatically locate available font files
regular_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",

        weight="normal"
    )
)

bold_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",
        weight="bold"
    )
)

# Typography hierarchy
title_font = ImageFont.truetype(bold_font_path, 36)
year_font = ImageFont.truetype(bold_font_path, 88)
subtitle_font = ImageFont.truetype(regular_font_path, 22)
legend_font = ImageFont.truetype(bold_font_path, 23)
label_font = ImageFont.truetype(regular_font_path, 19)
small_font = ImageFont.truetype(regular_font_path, 16)


In [ ]:
from matplotlib import font_manager
from PIL import ImageFont

regular_font_path = font_manager.findfont("DejaVu Sans")
bold_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",
        weight="bold"
    )
)

title_font = ImageFont.truetype(bold_font_path, 36)
year_font = ImageFont.truetype(bold_font_path, 88)
subtitle_font = ImageFont.truetype(regular_font_path, 22)
legend_font = ImageFont.truetype(bold_font_path, 23)
label_font = ImageFont.truetype(regular_font_path, 19)
small_font = ImageFont.truetype(regular_font_path, 16)


In [ ]:

# LAGOS VEGETATION MONITORING: PRESENTATION FRAME GENERATOR
# Study period: 2019–2025

import ee
import requests
import numpy as np
import matplotlib.colors as mcolors

from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
from matplotlib import font_manager



#FONT LOADING

regular_font_path = font_manager.findfont("DejaVu Sans")

bold_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",
        weight="bold"
    )
)

title_font = ImageFont.truetype(bold_font_path, 34)
year_font = ImageFont.truetype(bold_font_path, 88)
subtitle_font = ImageFont.truetype(regular_font_path, 21)
legend_font = ImageFont.truetype(bold_font_path, 23)
label_font = ImageFont.truetype(regular_font_path, 18)
small_font = ImageFont.truetype(regular_font_path, 17)

print("Typography loaded successfully.")


#NDVI VISUALISATION SETTINGS


palette = [
    "#a50026",
    "#fdae61",
    "#ffffbf",
    "#a6d96a",
    "#1a9850"
]

ndvi_vis = {
    "min": -0.2,
    "max": 0.8,
    "palette": palette
}



# RETRIEVE ANNUAL MAP FROM EARTH ENGINE


def get_annual_map(year):

    image = ee.Image(
        annual_composites
        .filter(ee.Filter.eq("year", year))
        .first()
    ).select("NDVI")

    visual = image.visualize(**ndvi_vis)

    boundaries = lagos_lgas.style(
        color="#555555",
        fillColor="00000000",
        width=1
    )

    white_background = ee.Image.constant(
        [255, 255, 255]
    ).byte()

    final_image = white_background.blend(
        visual.blend(boundaries)
    )

    url = final_image.getThumbURL({
        "region": lagos_geometry.bounds(),
        "dimensions": 1400,
        "format": "png"
    })

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    return Image.open(
        BytesIO(response.content)
    ).convert("RGB")



# CREATE PRESENTATION FRAME

def create_presentation_frame(map_image, year):

    canvas = Image.new(
        "RGB",
        (1600, 900),
        "white"
    )

    draw = ImageDraw.Draw(canvas)

    # Main title
    draw.text(
        (800, 45),
        "LAGOS STATE VEGETATION GREENNESS MONITORING",
        font=title_font,
        fill="#17324D",
        anchor="mt"
    )

    # Subtitle
    draw.text(
        (800, 105),
        "Sentinel-2 | Annual NDVI Assessment | 2019–2025",
        font=subtitle_font,
        fill="#606B75",
        anchor="mt"
    )

    # Dynamic year background
    draw.rounded_rectangle(
        (650, 150, 950, 265),
        radius=18,
        fill="#EAF3ED"
    )

    # Dynamic year label
    draw.text(
        (800, 162),
        str(year),
        font=year_font,
        fill="#176B3A",
        anchor="mt"
    )

    # Resize map without distorting its geographical shape
    map_copy = map_image.copy()

    map_copy.thumbnail((1460, 410))

    x = (1600 - map_copy.width) // 2
    y = 285

    canvas.paste(map_copy, (x, y))

    # Legend heading
    draw.text(
        (800, 725),
        "VEGETATION GREENNESS (NDVI)",
        font=legend_font,
        fill="#17324D",
        anchor="mt"
    )

    # Continuous colour gradient
    cmap = mcolors.LinearSegmentedColormap.from_list(
        "vegetation",
        palette
    )

    bar_x = 300
    bar_y = 775
    bar_width = 1000
    bar_height = 26

    for i in range(bar_width):

        colour = cmap(i / (bar_width - 1))

        rgb = tuple(
            int(v * 255)
            for v in colour[:3]
        )

        draw.line(
            (
                bar_x + i,
                bar_y,
                bar_x + i,
                bar_y + bar_height
            ),
            fill=rgb
        )

    # NDVI scale labels
    scale_values = [-0.2, 0.0, 0.2, 0.4, 0.6, 0.8]

    for value in scale_values:

        position = bar_x + int(
            ((value + 0.2) / 1.0) * bar_width
        )

        draw.text(
            (position, 813),
            f"{value:.1f}",
            font=label_font,
            fill="#333333",
            anchor="mt"
        )

    # Plain-language explanation
    draw.text(
        (800, 855),
        "Red: Low greenness     Yellow: Moderate     Green: High greenness",
        font=small_font,
        fill="#555555",
        anchor="mt"
    )

    return canvas



# GENERATE ALL SEVEN ANNUAL PRESENTATION FRAMES

presentation_frames = []

for year in range(2019, 2026):

    print(f"Processing {year}...")

    annual_map = get_annual_map(year)

    frame = create_presentation_frame(
        annual_map,
        year
    )

    presentation_frames.append(frame)



# VALIDATE AND PREVIEW

print("\nTotal presentation frames:", len(presentation_frames))

print(
    "Frame dimensions:",
    presentation_frames[0].size
)

print("\n2019 Preview:")
display(presentation_frames[0])

print("\n2025 Preview:")
display(presentation_frames[-1])

In [ ]:
import os
import numpy as np
import imageio.v2 as imageio
import shutil

# Confirm that all seven frames are available
if len(presentation_frames) != 7:
    raise ValueError(
        f"Expected 7 frames (2019–2025), but found {len(presentation_frames)}."
    )

# Convert frames to RGB arrays and hold each year for 2 seconds
video_frames = []

for frame in presentation_frames:
    rgb_frame = np.array(frame.convert("RGB"))
    video_frames.extend([rgb_frame, rgb_frame])

# Create the MP4
video_path = "/content/Lagos_NDVI_Animation_2019_2025_Enhanced.mp4"

imageio.mimsave(
    video_path,
    video_frames,
    fps=1,
    codec="libx264",
    quality=8,
    macro_block_size=16
)

# Verify the output
if not os.path.exists(video_path):
    raise FileNotFoundError("MP4 creation failed.")

file_size = os.path.getsize(video_path) / (1024 * 1024)

print("MP4 successfully created.")
print("File:", video_path)
print("Duration: approximately 14 seconds")
print("Resolution:", presentation_frames[0].size)
print(f"File size: {file_size:.2f} MB")

# Save a copy to Google Drive
from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

drive_folder = "/content/drive/MyDrive/Lagos_Vegetation_Monitoring"
os.makedirs(drive_folder, exist_ok=True)

drive_path = os.path.join(
    drive_folder,
    "Lagos_NDVI_Animation_2019_2025_Enhanced.mp4"
)

shutil.copy2(video_path, drive_path)

print("\nSaved to Google Drive:")
print(drive_path)